# 06 — Datasets, minibatches, and preprocessing

**Level:** Beginner → intermediate · **Study time:** 60–75 minutes (including experiments)

**Prerequisites:** Notebooks 04–05.

## What you will be able to do
- Implement a Dataset.
- Fit normalization on training data only.
- Train with shuffled minibatches and a correctly weighted epoch loss.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 05](05_neural_networks.ipynb) · [Next: 07](07_classification.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Split before estimating preprocessing statistics

A **minibatch** is a small group of observations used for one update. Before batching, create train/validation splits and estimate normalization from training data only. Validation statistics must not leak into preprocessing.


In [ ]:
raw_X = torch.randn(500, 3) * torch.tensor([1., 10., 100.]) + torch.tensor([5., -4., 20.])
y = raw_X @ torch.tensor([[2.], [0.3], [-0.02]]) + 0.2 * torch.randn(500, 1)
indices = torch.randperm(500)
train_idx, val_idx = indices[:400], indices[400:]
mean = raw_X[train_idx].mean(0)
std = raw_X[train_idx].std(0).clamp_min(1e-6)
X = (raw_X - mean) / std
assert torch.allclose(X[train_idx].mean(0), torch.zeros(3), atol=1e-5)


## Step 3 — Implement the two Dataset methods

A Dataset answers 'How many examples?' and 'What is example i?'. A DataLoader handles batching and shuffling. `num_workers=0` keeps these small examples portable in Windows notebooks.


In [ ]:
from torch.utils.data import Dataset

class RegressionDataset(Dataset):
    def __init__(self, features, targets):
        assert len(features) == len(targets)
        self.features, self.targets = features, targets
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.targets[index]

train_ds = RegressionDataset(X[train_idx], y[train_idx])
val_ds = RegressionDataset(X[val_idx], y[val_idx])
loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0,
                    generator=torch.Generator().manual_seed(SEED))
xb, yb = next(iter(loader))
print("One batch:", xb.shape, yb.shape, "Batches:", len(loader))
assert xb.shape == (32, 3)


## Step 4 — Train and aggregate the epoch loss

The last batch can be smaller. Multiply each batch's mean loss by its number of examples before summing; divide by the number of examples at the end. A plain mean of batch means would overweight a small final batch.


In [ ]:
model = nn.Linear(3, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.03)
epoch_losses = []
for epoch in range(40):
    model.train()
    total, count = 0., 0
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        count += len(xb)
    epoch_losses.append(total / count)
model.eval()
with torch.no_grad():
    val_mse = nn.functional.mse_loss(model(val_ds.features), val_ds.targets).item()
print("Final train MSE:", epoch_losses[-1], "Validation MSE:", val_mse)
assert val_mse < 0.2


## Your turn — practice before opening the solutions

1. With 400 training examples and batch size 64, predict how many batches and the final batch size.
2. Create a validation loader without shuffling.
3. Compute validation MSE by accumulating per-example squared errors, then compare with `val_mse`.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
alternative = DataLoader(train_ds, batch_size=64, shuffle=False)
sizes = [len(xb) for xb, yb in alternative]
assert len(sizes) == 7 and sizes[-1] == 16
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
total, count = 0., 0
with torch.no_grad():
    for xb, yb in val_loader:
        total += (model(xb) - yb).square().sum().item()
        count += len(xb)
assert abs(total / count - val_mse) < 1e-5


## Before moving on

Data preparation is part of the learned system. Save its training statistics with the model and apply the same transformation to new observations.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Datasets and DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)


[Course guide](../README.md) · [Previous: 05](05_neural_networks.ipynb) · [Next: 07](07_classification.ipynb)
